# 06. External-validation comparison and frozen contract

This notebook replaces the exploratory comparison notebook with a **paper-ready validation contract**. It performs no model selection on external outcomes.

Primary choices are frozen before external performance is calculated:

- the exact CCLE test split is recovered from the prespecified `res_enet_zbig`/`res_xgb_zbig` model inventories and verified across every primary HDF5 file;
- the only multigene models are z-score, $k=5000$ ElasticNet and XGBoost;
- the only primary same-gene baseline is a univariate linear model fitted on the exact CCLE training partition;
- external cohorts remain 20 independent cohorts grouped only by the recorded platform (`TMT`, `iTRAQ`, or label-free);
- raw/no-clipping RNA and native external protein values define the primary analysis;
- clipping, sample-scale alignment, target-scale harmonization, and the historical TMT1/TMT2 label are sensitivity analyses only;
- external measured protein values are never imputed.

The output is one immutable manifest plus one compact cohort contract per cohort. Every later table, figure, prediction checkpoint, and bootstrap result must contain the same manifest hash.


In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "6_normalization"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys

# Bound BLAS/OpenMP/XGBoost threading before importing numerical libraries.
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_thread_var, "1")

# Find the bundled shared module whether the kernel starts in the project root,
# notebooks/, notebooks2/, or the delivered package directory.
search_roots = []
if os.environ.get("PAPER_VALIDATION_MODULE_DIR"):
    search_roots.append(Path(os.environ["PAPER_VALIDATION_MODULE_DIR"]).expanduser().resolve())
cwd = Path.cwd().resolve()
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "paper_ready_external_validation", base / "notebooks", base / "notebooks2"]:
        if candidate not in search_roots:
            search_roots.append(candidate)
for candidate in search_roots:
    if (candidate / "paper_validation.py").exists():
        sys.path.insert(0, str(candidate))
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "paper_validation.py was not found. Keep it beside the five notebooks or set "
        "PAPER_VALIDATION_MODULE_DIR to the delivered package directory."
    )

import joblib
import numpy as np
import pandas as pd
import paper_validation as pv

PROJECT_ROOT = pv.detect_project_root()
RUN_ROOT = Path(os.environ.get(
    "PAPER_VALIDATION_RUN_ROOT",
    PROJECT_ROOT / "results_external_validation_paper_v1",
)).expanduser().resolve()
LAYOUT = pv.make_output_layout(RUN_ROOT)
MANIFEST_PATH = RUN_ROOT / "validation_manifest.json"
SMOKE_TEST = os.environ.get("PAPER_VALIDATION_SMOKE_TEST", "0") == "1"

print("Notebook 06")
print("Shared module:", MODULE_DIR / "paper_validation.py")
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Smoke-test mode:", SMOKE_TEST)


## 1. Configuration


In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data"   # single 06. validation/data folder
HASH_MODEL_FILES = os.environ.get("PAPER_VALIDATION_HASH_MODELS", "1" if not SMOKE_TEST else "0") == "1"
VERIFY_ALL_MODEL_SPLITS = True
MAX_SAMPLE_MISSING = 0.50
MAX_RNA_FEATURE_MISSING = 0.50
MAX_PROTEIN_FEATURE_MISSING = 0.50
CLIP_QUANTILES = (0.005, 0.995)
EXPECTED_SPLIT_HASH = os.environ.get("PAPER_VALIDATION_EXPECTED_SPLIT_HASH", "").strip() or None

EXPECTED_CCLE_TOTAL = None if SMOKE_TEST else 369
EXPECTED_CCLE_TEST = None if SMOKE_TEST else 74
EXPECTED_CCLE_TRAIN = None if SMOKE_TEST else 295
EXPECTED_PROTEINS = None if SMOKE_TEST else 5113
EXPECTED_COHORTS = 20

print("Processed data:", PROCESSED_DIR)
print("Hash all primary model files:", HASH_MODEL_FILES)
print("Primary model specifications:", pv.MODEL_SPECS)
print("Expected cohorts:", EXPECTED_COHORTS)
print("Externally frozen split hash:", EXPECTED_SPLIT_HASH or "not supplied; all selected artifacts must agree")


## 2. Reconcile the CCLE split and verify the exact primary model inventories


In [ ]:
ccle_rna, ccle_protein = pv.load_ccle_matrices(PROCESSED_DIR)
if EXPECTED_CCLE_TOTAL is not None and len(ccle_rna) != EXPECTED_CCLE_TOTAL:
    raise RuntimeError(f"Expected {EXPECTED_CCLE_TOTAL} matched CCLE samples, found {len(ccle_rna)}")
if EXPECTED_PROTEINS is not None and ccle_protein.shape[1] != EXPECTED_PROTEINS:
    raise RuntimeError(f"Expected {EXPECTED_PROTEINS} CCLE proteins, found {ccle_protein.shape[1]}")

model_inventory, split_info = pv.build_primary_model_inventory(
    PROJECT_ROOT,
    ccle_protein.columns.astype(str),
    expected_sample_ids=ccle_rna.index.astype(str),
    compute_file_hashes=HASH_MODEL_FILES,
    verify_all_splits=VERIFY_ALL_MODEL_SPLITS,
    progress_every=500,
)

if EXPECTED_SPLIT_HASH is not None and split_info["test_split_hash_sorted"] != EXPECTED_SPLIT_HASH:
    raise RuntimeError(
        f"Verified model split {split_info['test_split_hash_sorted']} does not match "
        f"PAPER_VALIDATION_EXPECTED_SPLIT_HASH={EXPECTED_SPLIT_HASH}"
    )

if EXPECTED_CCLE_TEST is not None and split_info["test_sample_count"] != EXPECTED_CCLE_TEST:
    raise RuntimeError(
        f"Expected {EXPECTED_CCLE_TEST} held-out samples, found {split_info['test_sample_count']}"
    )

print("Canonical test hash (sorted):", split_info["test_split_hash_sorted"])
print("Canonical train hash (sorted):", split_info["train_split_hash_sorted"])
print("Canonical test source:", split_info["test_split_location"])
print("Canonical train source:", split_info["train_split_location"])
print("Canonical model:", split_info["canonical_model_path"])
print(model_inventory.groupby("display_model")["protein"].nunique())


## 3. Fit the CCLE-only reference and the single primary same-gene baseline


In [ ]:
reference = pv.fit_ccle_reference(
    ccle_rna,
    ccle_protein,
    split_info["test_samples"],
    clip_quantiles=CLIP_QUANTILES,
)
proxy_parameters = pv.fit_linear_proxy_parameters(reference)

if EXPECTED_CCLE_TRAIN is not None and len(reference["train_samples"]) != EXPECTED_CCLE_TRAIN:
    raise RuntimeError(
        f"Expected {EXPECTED_CCLE_TRAIN} CCLE training samples, found {len(reference['train_samples'])}"
    )
if reference["test_split_hash_sorted"] != split_info["test_split_hash_sorted"]:
    raise RuntimeError("Reference test-split hash differs from the verified primary-model split hash.")
reference_train_hash = pv.split_hash(reference["train_samples"], sorted_values=True)
if reference_train_hash != split_info["train_split_hash_sorted"]:
    raise RuntimeError("Reference train-split hash differs from the verified primary-model split hash.")

print("CCLE training samples:", len(reference["train_samples"]))
print("CCLE held-out samples:", len(reference["test_samples"]))
print("RNA features:", len(reference["rna_columns"]))
print("Proteins:", len(reference["protein_columns"]))
print("Primary proxy-eligible proteins:", int(proxy_parameters["same_gene_available"].sum()))


## 4. Build 20 independent paired cohort contracts


In [ ]:
cohort_paths = {}
cohort_hashes = {}
audit_rows = []

for platform in ["TMT", "iTRAQ", "LabelFree"]:
    print(f"Loading {platform}...")
    rna_platform, protein_platform = pv.load_external_platform(PROCESSED_DIR, platform)
    observed_rna = {pv.infer_cohort(x, platform) for x in rna_platform.index}
    observed_protein = {pv.infer_cohort(x, platform) for x in protein_platform.index}
    expected = set(pv.EXPECTED_COHORTS_BY_PLATFORM[platform])
    unknown = sorted((observed_rna | observed_protein) - expected)
    missing = sorted(expected - (observed_rna & observed_protein))
    if unknown or missing:
        raise RuntimeError(
            f"{platform} cohort labels do not match the frozen 20-cohort specification. "
            f"unknown={unknown}, missing={missing}"
        )
    for cohort in pv.EXPECTED_COHORTS_BY_PLATFORM[platform]:
        bundle, audit = pv.build_cohort_bundle(
            platform,
            cohort,
            rna_platform,
            protein_platform,
            reference,
            max_sample_missing=MAX_SAMPLE_MISSING,
            max_rna_feature_missing=MAX_RNA_FEATURE_MISSING,
            max_protein_feature_missing=MAX_PROTEIN_FEATURE_MISSING,
        )
        path = LAYOUT["cohorts"] / f"{pv.safe_name(cohort)}.joblib"
        pv.atomic_joblib_dump(bundle, path, compress=3)
        cohort_paths[cohort] = str(path.resolve())
        cohort_hashes[cohort] = pv.file_sha256(path)
        audit_rows.append(audit)
        print(f"  {cohort:20s}: {audit['samples_retained']:4d} samples, {audit['proteins_retained']:5d} proteins")
        del bundle
    del rna_platform, protein_platform
    gc.collect()

cohort_audit = pd.DataFrame(audit_rows).sort_values(["platform", "cohort"]).reset_index(drop=True)
if len(cohort_audit) != EXPECTED_COHORTS:
    raise RuntimeError(f"Expected {EXPECTED_COHORTS} cohort contracts, found {len(cohort_audit)}")
if int(cohort_audit["samples_retained"].sum()) <= 0:
    raise RuntimeError("No external samples were retained.")
cohort_audit_path = LAYOUT["contract"] / "cohort_qc_audit.csv"
cohort_audit.to_csv(cohort_audit_path, index=False)
cohort_audit



## 5. Normalization QC: before/after distributions

This section visualizes the exact transformations used by the paper-ready validation pipeline without changing the frozen analysis contract.

- **RNA primary path:** external RNA after identifier harmonization and CCLE-training median filling, followed by the **saved CCLE model normalizer**. No external clipping or distribution alignment is used in the primary analysis.
- **RNA sensitivities:** CCLE-training quantile clipping and robust sample-scale alignment are shown separately, each followed by the same saved CCLE normalizer.
- **Protein primary target:** native external protein values are left unchanged. The ``harmonized`` protein target is shown only as a sensitivity diagnostic.

The histograms use deterministic subsampling only for visualization; all QC percentages are calculated from the complete cohort matrices. Plot-axis trimming is display-only and does not alter any analysis values.

### Primary model/evaluation-path distribution grid

A fourth diagnostic figure is added specifically for the **data entering the primary frozen models and the native protein targets used to score them**. For RNA, the ``before`` panel is the exact full CCLE model schema after missing external genes have been filled with CCLE-training medians but **before** the saved CCLE normalizer; the ``after`` panel is the same matrix after that frozen normalizer. Protein-specific models subsequently select their saved 5,000 features from this normalized schema. For proteins, the primary target is native and is not normalized, so the before/after target distributions are intentionally identical; the duplicate panel is retained to make that identity transformation explicit.


A second, title-free version of the primary model/evaluation-path grid is also saved with substantially larger typography. It uses one shared density label, one row label per platform, one column header per data stage, and x-axis labels only on the bottom row.


All figures are written as vector PDFs under `figures_pdf/`; typography is doubled and histogram curves are thicker.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gc
import joblib
import os

FIGURES_PDF_DIR = FIG_DIR
FIGURES_PDF_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZATION_FIG_DIR = FIGURES_PDF_DIR

PLOT_MAX_VALUES_PER_COHORT = int(
    os.environ.get("PAPER_VALIDATION_QC_PLOT_VALUES", "80000")
)
PLOT_SEED = 2026

# -------------------------------------------------------------------------
# TYPOGRAPHY & SPACING SETTINGS
# Applied consistently across all figures
# -------------------------------------------------------------------------
# Every plotting font is exactly doubled relative to the previous version.
RNA_FIGSIZE = (40, 24)

RNA_AXIS_FONT = 36          # was 18
RNA_TICK_FONT = 30          # was 15
RNA_HEADER_FONT = 38        # was 19
RNA_ROW_FONT = 38           # was 19
RNA_LEGEND_FONT = 28        # was 14
RNA_LINEWIDTH = 4.0         # was 2; thicker publication curves

# Larger canvas + tight bounding boxes keep the doubled typography inside the PDF.
ROW_LABEL_OFFSET = 0.030
DENSITY_LABEL_OFFSET = 0.064

plt.rcParams.update({
    "font.size": 28,
    "axes.labelsize": RNA_AXIS_FONT,
    "axes.titlesize": RNA_HEADER_FONT,
    "xtick.labelsize": RNA_TICK_FONT,
    "ytick.labelsize": RNA_TICK_FONT,
    "legend.fontsize": RNA_LEGEND_FONT,
    "lines.linewidth": RNA_LINEWIDTH,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})


# Use the normalizer stored in the verified canonical primary model artifact.
canonical_artifact = pv.load_h5_artifact(
    split_info["canonical_model_path"]
)
canonical_normalizer = canonical_artifact["normalizer"]

canonical_columns = pd.Index(
    [
        str(x)
        for x in canonical_normalizer.get("columns", [])
    ]
)

if not canonical_columns.equals(
    pd.Index(reference["rna_columns"]).astype(str)
):
    raise RuntimeError(
        "Canonical saved normalizer columns do not match "
        "the frozen CCLE RNA schema."
    )


# Verify the saved normalizer is compatible with the CCLE reference statistics.
ccle_train_rna = reference["rna_train"].reindex(
    columns=canonical_columns
)

ccle_train_rna_normalized = pv.apply_saved_normalizer(
    ccle_train_rna,
    canonical_normalizer,
)


def _finite_sample(values, max_values, seed):
    arr = (
        values.to_numpy(dtype=np.float32, copy=False)
        if isinstance(values, pd.DataFrame)
        else np.asarray(values, dtype=np.float32)
    )

    flat = arr.reshape(-1)
    idx = np.flatnonzero(np.isfinite(flat))

    if len(idx) == 0:
        return np.asarray([], dtype=np.float32)

    if len(idx) > max_values:
        rng = np.random.default_rng(seed)
        idx = rng.choice(
            idx,
            size=max_values,
            replace=False,
        )

    return flat[idx].astype(
        np.float32,
        copy=False,
    )


def _plot_hist(
    ax,
    series_list,
    labels,
    _unused_title=None,
    bins=80,
    linewidth=RNA_LINEWIDTH,
):
    """Draw a title-free density histogram on a robust common range."""

    nonempty = [
        np.asarray(x, dtype=float)
        for x in series_list
        if len(x)
    ]

    if not nonempty:
        ax.text(
            0.5,
            0.5,
            "No finite values",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=RNA_AXIS_FONT,
        )
        ax.grid(alpha=0.15)
        return

    merged = np.concatenate(nonempty)

    lo, hi = np.nanquantile(
        merged,
        [0.01, 0.99],
    )

    if (
        not np.isfinite(lo)
        or not np.isfinite(hi)
        or lo >= hi
    ):
        lo, hi = (
            np.nanmin(merged),
            np.nanmax(merged),
        )

    if (
        not np.isfinite(lo)
        or not np.isfinite(hi)
        or lo >= hi
    ):
        lo, hi = -1.0, 1.0

    for values, label in zip(
        series_list,
        labels,
    ):
        values = np.asarray(
            values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
        ]

        if len(values):
            ax.hist(
                values,
                bins=bins,
                range=(lo, hi),
                density=True,
                histtype="step",
                linewidth=linewidth,
                label=label,
            )

    ax.grid(
        alpha=0.15,
        linewidth=1.6,
    )


def _format_hist_grid(
    fig,
    axes,
    *,
    row_labels,
    column_labels,
    x_labels,
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    row_label_offset=ROW_LABEL_OFFSET,
    density_label_offset=DENSITY_LABEL_OFFSET,
):
    """Apply large, non-redundant labels and title-free row/column headers."""

    axes = np.asarray(axes)
    n_rows, n_cols = axes.shape

    # --------------------------------------------------------------
    # Axis labels and numerical tick labels
    # --------------------------------------------------------------
    for row in range(n_rows):
        for col in range(n_cols):
            ax = axes[row, col]

            ax.set_ylabel("")

            ax.set_xlabel(
                x_labels[col]
                if row == n_rows - 1
                else "",
                fontsize=axis_font,
                labelpad=10,
            )

            ax.tick_params(
                axis="both",
                labelsize=tick_font,
                pad=8,
                width=1.6,
                length=7,
            )

    # --------------------------------------------------------------
    # Column headers
    # --------------------------------------------------------------
    for col, label in enumerate(
        column_labels
    ):
        axes[0, col].text(
            0.5,
            1.085,
            label,
            transform=axes[0, col].transAxes,
            ha="center",
            va="bottom",
            fontsize=header_font,
            fontweight="bold",
            clip_on=False,
        )

    # --------------------------------------------------------------
    # Row labels and common y-axis label (positioned to avoid tick overlap)
    # --------------------------------------------------------------
    fig.canvas.draw()

    left_edge = min(
        ax.get_position().x0
        for ax in axes[:, 0]
    )

    for row, label in enumerate(
        row_labels
    ):
        position = axes[
            row,
            0,
        ].get_position()

        fig.text(
            left_edge - row_label_offset,
            0.5 * (
                position.y0
                + position.y1
            ),
            label,
            ha="center",
            va="center",
            rotation=90,
            fontsize=row_font,
            fontweight="bold",
        )

    fig.text(
        left_edge - density_label_offset,
        0.5,
        "Density",
        ha="center",
        va="center",
        rotation=90,
        fontsize=axis_font,
        fontweight="normal",
    )


# -------------------------------------------------------------------------
# Small deterministic CCLE plotting references
# -------------------------------------------------------------------------
ccle_raw_plot = _finite_sample(
    ccle_train_rna,
    PLOT_MAX_VALUES_PER_COHORT,
    PLOT_SEED,
)

ccle_norm_plot = _finite_sample(
    ccle_train_rna_normalized,
    PLOT_MAX_VALUES_PER_COHORT,
    PLOT_SEED + 1,
)

ccle_protein_plot = _finite_sample(
    reference["protein_train"],
    PLOT_MAX_VALUES_PER_COHORT,
    PLOT_SEED + 2,
)


rna_plot_values = {
    platform: {
        stage: []
        for stage in (
            "before",
            "primary_pre_normalizer",
            "primary",
            "clipped",
            "scale_aligned",
        )
    }
    for platform in [
        "TMT",
        "iTRAQ",
        "LabelFree",
    ]
}

protein_plot_values = {
    group: {
        stage: []
        for stage in (
            "raw",
            "harmonized",
        )
    }
    for group in [
        "TMT1",
        "TMT2",
        "iTRAQ",
        "LabelFree",
    ]
}

normalization_audit_frames = []
protein_change_rows = []


for cohort_idx, cohort in enumerate(
    pv.COHORT_TABLE["cohort"].astype(str)
):
    bundle = joblib.load(
        cohort_paths[cohort]
    )

    platform = str(
        bundle["platform"]
    )

    variants, rna_audit = (
        pv.prepare_rna_variants(
            bundle,
            reference,
        )
    )

    targets = pv.prepare_protein_targets(
        bundle,
        reference,
    )

    # BEFORE
    rna_plot_values[
        platform
    ]["before"].append(
        _finite_sample(
            bundle["rna_raw"],
            PLOT_MAX_VALUES_PER_COHORT,
            PLOT_SEED
            + 10
            + cohort_idx,
        )
    )

    # PRIMARY PRE-NORMALIZER
    rna_plot_values[
        platform
    ]["primary_pre_normalizer"].append(
        _finite_sample(
            variants["raw"],
            PLOT_MAX_VALUES_PER_COHORT,
            PLOT_SEED
            + 50
            + cohort_idx,
        )
    )

    # AFTER
    for stage in (
        "primary",
        "clipped",
        "scale_aligned",
    ):
        variant_key = {
            "primary": "raw",
            "clipped": "clipped",
            "scale_aligned": "scale_aligned",
        }[stage]

        normalized = (
            pv.apply_saved_normalizer(
                variants[variant_key],
                canonical_normalizer,
            )
        )

        rna_plot_values[
            platform
        ][stage].append(
            _finite_sample(
                normalized,
                PLOT_MAX_VALUES_PER_COHORT,
                PLOT_SEED
                + 100
                + cohort_idx,
            )
        )

        del normalized

    normalization_audit_frames.append(
        rna_audit
    )

    scale_group = str(
        bundle.get(
            "tmt_scale_sensitivity",
            "not_applicable",
        )
    )

    protein_group = (
        scale_group
        if (
            platform == "TMT"
            and scale_group
            in {"TMT1", "TMT2"}
        )
        else platform
    )

    raw_protein = targets["raw"]
    harmonized_protein = targets[
        "harmonized"
    ]

    protein_plot_values[
        protein_group
    ]["raw"].append(
        _finite_sample(
            raw_protein,
            PLOT_MAX_VALUES_PER_COHORT,
            PLOT_SEED
            + 200
            + cohort_idx,
        )
    )

    protein_plot_values[
        protein_group
    ]["harmonized"].append(
        _finite_sample(
            harmonized_protein,
            PLOT_MAX_VALUES_PER_COHORT,
            PLOT_SEED
            + 300
            + cohort_idx,
        )
    )

    raw_arr = raw_protein.to_numpy(
        dtype=np.float32,
        copy=False,
    )

    harm_arr = (
        harmonized_protein.to_numpy(
            dtype=np.float32,
            copy=False,
        )
    )

    finite = (
        np.isfinite(raw_arr)
        & np.isfinite(harm_arr)
    )

    protein_change_rows.append(
        {
            "cohort": cohort,
            "platform": platform,
            "tmt_scale_sensitivity": scale_group,
            "protein_harmonized_values_changed_pct": float(
                100.0
                * np.sum(
                    finite
                    & ~np.isclose(
                        raw_arr,
                        harm_arr,
                        equal_nan=True,
                    )
                )
                / max(
                    1,
                    finite.sum(),
                )
            ),
        }
    )

    del (
        bundle,
        variants,
        targets,
        raw_protein,
        harmonized_protein,
    )

    gc.collect()


normalization_qc_summary = (
    pd.concat(
        normalization_audit_frames,
        ignore_index=True,
    )
    .merge(
        pd.DataFrame(
            protein_change_rows
        ),
        on=[
            "cohort",
            "platform",
            "tmt_scale_sensitivity",
        ],
        how="left",
    )
    .sort_values(
        [
            "platform",
            "cohort",
        ]
    )
    .reset_index(
        drop=True
    )
)

normalization_qc_summary_path = (
    LAYOUT["contract"]
    / "normalization_qc_summary.csv"
)

normalization_qc_summary.to_csv(
    normalization_qc_summary_path,
    index=False,
)


# -------------------------------------------------------------------------
# Concatenate deterministic plotting samples
# -------------------------------------------------------------------------
for platform in rna_plot_values:
    for stage in rna_plot_values[
        platform
    ]:
        chunks = rna_plot_values[
            platform
        ][stage]

        rna_plot_values[
            platform
        ][stage] = (
            np.concatenate(chunks)
            if chunks
            else np.asarray(
                [],
                dtype=np.float32,
            )
        )


for group in protein_plot_values:
    for stage in protein_plot_values[
        group
    ]:
        chunks = protein_plot_values[
            group
        ][stage]

        protein_plot_values[
            group
        ][stage] = (
            np.concatenate(chunks)
            if chunks
            else np.asarray(
                [],
                dtype=np.float32,
            )
        )


# =========================================================================
# FIGURE 1: RNA Normalization QC
# =========================================================================
fig, axes = plt.subplots(
    3,
    4,
    figsize=RNA_FIGSIZE,
    constrained_layout=True,
)

for row, platform in enumerate(
    [
        "TMT",
        "iTRAQ",
        "LabelFree",
    ]
):
    _plot_hist(
        axes[row, 0],
        [
            ccle_raw_plot,
            rna_plot_values[
                platform
            ]["before"],
        ],
        [
            "CCLE train",
            f"{platform} external",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    _plot_hist(
        axes[row, 1],
        [
            ccle_norm_plot,
            rna_plot_values[
                platform
            ]["primary"],
        ],
        [
            "CCLE train after\nsaved normalizer",
            f"{platform} primary",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    _plot_hist(
        axes[row, 2],
        [
            ccle_norm_plot,
            rna_plot_values[
                platform
            ]["clipped"],
        ],
        [
            "CCLE train after\nsaved normalizer",
            f"{platform} clipped sensitivity",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    _plot_hist(
        axes[row, 3],
        [
            ccle_norm_plot,
            rna_plot_values[
                platform
            ]["scale_aligned"],
        ],
        [
            "CCLE train after\nsaved normalizer",
            f"{platform} scale-aligned sensitivity",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    # Legends in the first row
    for col in range(4):
        if row == 0:
            axes[
                row,
                col,
            ].legend(
                fontsize=RNA_LEGEND_FONT,
                frameon=True,
                borderpad=0.6,
                labelspacing=0.45,
                handlelength=2.0,
            )


_format_hist_grid(
    fig,
    axes,
    row_labels=[
        "TMT",
        "iTRAQ",
        "Label-free",
    ],
    column_labels=[
        "RNA before normalization",
        "Primary RNA model input",
        "Clipping sensitivity",
        "Scale-alignment sensitivity",
    ],
    x_labels=[
        "Stored RNA value",
        "Model-input RNA value",
        "Model-input RNA value",
        "Model-input RNA value",
    ],
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    row_label_offset=ROW_LABEL_OFFSET,
    density_label_offset=DENSITY_LABEL_OFFSET,
)

rna_normalization_figure_path = (
    NORMALIZATION_FIG_DIR
    / "rna_normalization_before_after.pdf"
)

fig.savefig(
    rna_normalization_figure_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.45,
)

plt.show()
plt.close(fig)


# =========================================================================
# FIGURE 2: Protein Target Scale
# =========================================================================
fig, axes = plt.subplots(
    4,
    2,
    figsize=(30, 26),
    constrained_layout=True,
)

for row, group in enumerate(
    [
        "TMT1",
        "TMT2",
        "iTRAQ",
        "LabelFree",
    ]
):
    _plot_hist(
        axes[row, 0],
        [
            ccle_protein_plot,
            protein_plot_values[
                group
            ]["raw"],
        ],
        [
            "CCLE train protein",
            f"{group} raw",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    _plot_hist(
        axes[row, 1],
        [
            ccle_protein_plot,
            protein_plot_values[
                group
            ]["harmonized"],
        ],
        [
            "CCLE train protein",
            f"{group} harmonized sensitivity",
        ],
        linewidth=RNA_LINEWIDTH,
    )

    if row == 0:
        axes[
            row,
            0,
        ].legend(
            fontsize=RNA_LEGEND_FONT,
            frameon=True,
            borderpad=0.6,
        )

        axes[
            row,
            1,
        ].legend(
            fontsize=RNA_LEGEND_FONT,
            frameon=True,
            borderpad=0.6,
        )


_format_hist_grid(
    fig,
    axes,
    row_labels=[
        "TMT1",
        "TMT2",
        "iTRAQ",
        "Label-free",
    ],
    column_labels=[
        "Native primary target",
        "Scale-harmonization sensitivity",
    ],
    x_labels=[
        "Protein value",
        "Protein value",
    ],
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    row_label_offset=0.055,
    density_label_offset=0.105,
)

protein_normalization_figure_path = (
    NORMALIZATION_FIG_DIR
    / "protein_target_before_after_sensitivity.pdf"
)

fig.savefig(
    protein_normalization_figure_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.45,
)

plt.show()
plt.close(fig)


# =========================================================================
# FIGURE 3: Exact Fraction of Observed RNA Changed
# =========================================================================
plot_audit = (
    normalization_qc_summary.copy()
)

x = np.arange(
    len(plot_audit)
)

width = 0.38

fig, ax = plt.subplots(
    figsize=(32, 13),
    constrained_layout=True,
)

ax.bar(
    x - width / 2,
    plot_audit[
        "clipped_observed_values_pct"
    ],
    width,
    label="RNA changed by clipping",
)

ax.bar(
    x + width / 2,
    plot_audit[
        "scale_aligned_observed_values_changed_pct"
    ],
    width,
    label="RNA changed by scale alignment",
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_audit["cohort"],
    rotation=65,
    ha="right",
    fontsize=RNA_TICK_FONT,
)

ax.tick_params(
    axis="y",
    labelsize=RNA_TICK_FONT,
)

ax.set_ylabel(
    "Observed RNA values changed (%)",
    fontsize=RNA_AXIS_FONT,
    labelpad=10,
)

ax.legend(
    fontsize=RNA_LEGEND_FONT,
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.2,
)

rna_change_audit_figure_path = (
    NORMALIZATION_FIG_DIR
    / "rna_normalization_change_audit.pdf"
)

fig.savefig(
    rna_change_audit_figure_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.45,
)

plt.show()
plt.close(fig)


# =========================================================================
# FIGURE 4: PRIMARY MODEL/EVALUATION PATH ONLY
# =========================================================================
protein_primary_by_platform = {
    "TMT": np.concatenate(
        [
            protein_plot_values[
                "TMT1"
            ]["raw"],
            protein_plot_values[
                "TMT2"
            ]["raw"],
        ]
    ),
    "iTRAQ": protein_plot_values[
        "iTRAQ"
    ]["raw"],
    "LabelFree": protein_plot_values[
        "LabelFree"
    ]["raw"],
}


PRIMARY_GRID_ROWS = [
    "TMT",
    "iTRAQ",
    "LabelFree",
]

PRIMARY_GRID_ROW_LABELS = [
    "TMT",
    "iTRAQ",
    "Label-free",
]

PRIMARY_GRID_COLUMNS = [
    "RNA before\nfrozen normalizer",
    "RNA used by\nthe model",
    "Native protein\ntarget",
    "Primary evaluation\ntarget",
]

PRIMARY_GRID_XLABELS = [
    "RNA value before\nsaved normalizer",
    "RNA model-input\nvalue",
    "Native protein\nvalue",
    "Primary evaluation-target\nvalue",
]


def _render_primary_model_data_grid(
    *,
    path,
    figsize=RNA_FIGSIZE,
    dpi=220,
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    legend_font=RNA_LEGEND_FONT,
    annotation_font=26,
    linewidth=RNA_LINEWIDTH,
    row_label_offset=ROW_LABEL_OFFSET,
    density_label_offset=DENSITY_LABEL_OFFSET,
):
    fig, axes = plt.subplots(
        3,
        4,
        figsize=figsize,
        constrained_layout=True,
    )

    for row, platform in enumerate(
        PRIMARY_GRID_ROWS
    ):
        _plot_hist(
            axes[row, 0],
            [
                ccle_raw_plot,
                rna_plot_values[
                    platform
                ][
                    "primary_pre_normalizer"
                ],
            ],
            [
                "CCLE train RNA",
                f"{platform} primary\npre-normalizer",
            ],
            linewidth=linewidth,
        )

        _plot_hist(
            axes[row, 1],
            [
                ccle_norm_plot,
                rna_plot_values[
                    platform
                ]["primary"],
            ],
            [
                "CCLE train after\nsaved normalizer",
                f"{platform} primary\nmodel input",
            ],
            linewidth=linewidth,
        )

        protein_values = (
            protein_primary_by_platform[
                platform
            ]
        )

        _plot_hist(
            axes[row, 2],
            [
                ccle_protein_plot,
                protein_values,
            ],
            [
                "CCLE train protein",
                f"{platform} native external\nprotein",
            ],
            linewidth=linewidth,
        )

        _plot_hist(
            axes[row, 3],
            [
                ccle_protein_plot,
                protein_values,
            ],
            [
                "CCLE train protein",
                f"{platform} primary target\n(unchanged)",
            ],
            linewidth=linewidth,
        )

        axes[
            row,
            3,
        ].text(
            0.98,
            0.96,
            "Identity target: no protein normalization",
            ha="right",
            va="top",
            transform=axes[
                row,
                3,
            ].transAxes,
            fontsize=annotation_font,
            bbox={
                "boxstyle": "round,pad=0.25",
                "facecolor": "white",
                "alpha": 0.78,
                "edgecolor": "0.7",
            },
        )

        for col in range(4):
            if row == 0:
                axes[
                    row,
                    col,
                ].legend(
                    fontsize=legend_font,
                    frameon=True,
                    borderpad=0.6,
                )

    _format_hist_grid(
        fig,
        axes,
        row_labels=PRIMARY_GRID_ROW_LABELS,
        column_labels=PRIMARY_GRID_COLUMNS,
        x_labels=PRIMARY_GRID_XLABELS,
        axis_font=axis_font,
        tick_font=tick_font,
        header_font=header_font,
        row_font=row_font,
        row_label_offset=row_label_offset,
        density_label_offset=density_label_offset,
    )

    fig.savefig(
        path,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.45,
    )

    plt.show()
    plt.close(fig)


primary_model_data_grid_path = (
    NORMALIZATION_FIG_DIR
    / "primary_model_data_before_after_grid.pdf"
)

_render_primary_model_data_grid(
    path=primary_model_data_grid_path,
    figsize=RNA_FIGSIZE,
    dpi=220,
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    legend_font=RNA_LEGEND_FONT,
    annotation_font=26,
    linewidth=RNA_LINEWIDTH,
    row_label_offset=ROW_LABEL_OFFSET,
    density_label_offset=DENSITY_LABEL_OFFSET,
)


primary_model_data_grid_large_text_path = (
    NORMALIZATION_FIG_DIR
    / "figure_external_validation_normalization_rna.pdf"
)

_render_primary_model_data_grid(
    path=primary_model_data_grid_large_text_path,
    figsize=RNA_FIGSIZE,
    dpi=300,
    axis_font=RNA_AXIS_FONT,
    tick_font=RNA_TICK_FONT,
    header_font=RNA_HEADER_FONT,
    row_font=RNA_ROW_FONT,
    legend_font=RNA_LEGEND_FONT,
    annotation_font=28,
    linewidth=RNA_LINEWIDTH,
    row_label_offset=ROW_LABEL_OFFSET,
    density_label_offset=DENSITY_LABEL_OFFSET,
)


normalization_figure_registry = pd.DataFrame(
    [
        {
            "figure": "rna_before_after",
            "path": str(
                rna_normalization_figure_path.resolve()
            ),
            "sha256": pv.file_sha256(
                rna_normalization_figure_path
            ),
            "primary_or_sensitivity": (
                "primary + sensitivity"
            ),
        },
        {
            "figure": "protein_before_after_sensitivity",
            "path": str(
                protein_normalization_figure_path.resolve()
            ),
            "sha256": pv.file_sha256(
                protein_normalization_figure_path
            ),
            "primary_or_sensitivity": (
                "primary + sensitivity"
            ),
        },
        {
            "figure": "rna_change_audit",
            "path": str(
                rna_change_audit_figure_path.resolve()
            ),
            "sha256": pv.file_sha256(
                rna_change_audit_figure_path
            ),
            "primary_or_sensitivity": (
                "sensitivity audit"
            ),
        },
        {
            "figure": "primary_model_data_before_after_grid",
            "path": str(
                primary_model_data_grid_path.resolve()
            ),
            "sha256": pv.file_sha256(
                primary_model_data_grid_path
            ),
            "primary_or_sensitivity": (
                "primary model/evaluation path"
            ),
        },
        {
            "figure": "primary_model_data_before_after_grid_large_text",
            "path": str(
                primary_model_data_grid_large_text_path.resolve()
            ),
            "sha256": pv.file_sha256(
                primary_model_data_grid_large_text_path
            ),
            "primary_or_sensitivity": (
                "primary model/evaluation path — "
                "large-text title-free version"
            ),
        },
    ]
)

normalization_figure_registry_path = (
    LAYOUT["contract"]
    / "normalization_figure_registry.csv"
)

normalization_figure_registry.to_csv(
    normalization_figure_registry_path,
    index=False,
)

display(
    normalization_qc_summary
)

display(
    normalization_figure_registry
)

## 6. Save the immutable contract and manifest


In [ ]:
reference_path = LAYOUT["contract"] / "ccle_reference.joblib"
proxy_base = LAYOUT["contract"] / "linear_proxy_parameters"
inventory_base = LAYOUT["contract"] / "primary_model_inventory"
pv.atomic_joblib_dump(reference, reference_path, compress=3)
proxy_path = pv.write_table(proxy_parameters, proxy_base, index=False)
inventory_path = pv.write_table(model_inventory, inventory_base, index=False)

files = {
    "ccle_reference": {"path": str(reference_path.resolve()), "sha256": pv.file_sha256(reference_path)},
    "linear_proxy_parameters": {"path": str(proxy_path.resolve()), "sha256": pv.file_sha256(proxy_path)},
    "model_inventory": {"path": str(inventory_path.resolve()), "sha256": pv.file_sha256(inventory_path)},
    "cohort_qc_audit": {"path": str(cohort_audit_path.resolve()), "sha256": pv.file_sha256(cohort_audit_path)},
    "normalization_qc_summary": {"path": str(normalization_qc_summary_path.resolve()), "sha256": pv.file_sha256(normalization_qc_summary_path)},
    "normalization_figure_registry": {"path": str(normalization_figure_registry_path.resolve()), "sha256": pv.file_sha256(normalization_figure_registry_path)},
    "normalization_figures": {
        row["figure"]: {"path": row["path"], "sha256": row["sha256"]}
        for _, row in normalization_figure_registry.iterrows()
    },
    "cohort_contracts": {
        cohort: {"path": cohort_paths[cohort], "sha256": cohort_hashes[cohort]}
        for cohort in pv.COHORT_TABLE["cohort"]
    },
}

manifest_core = {
    "pipeline_version": pv.PIPELINE_VERSION,
    "project_root": str(PROJECT_ROOT.resolve()),
    "run_root": str(RUN_ROOT.resolve()),
    "split": split_info,
    "model_inventory_hash": split_info["inventory_hash"],
    "externally_frozen_split_hash": EXPECTED_SPLIT_HASH,
    "primary_analysis": {
        "proxy": "CCLE-training univariate same-gene linear regression",
        "elasticnet": "res_enet_zbig | zscore | k5000",
        "xgboost": "res_xgb_zbig | zscore | k5000",
        "external_model_selection": False,
        "rna": "raw external RNA after ID harmonization; CCLE-training median fill; no clipping",
        "protein_target": "native/raw harmonized-collection protein values; no imputation",
        "unit": "20 independent cohorts",
        "platform_groups": ["TMT", "iTRAQ", "LabelFree"],
        "tmt1_tmt2_used": False,
        "external_sctranslator": "removed_incomplete",
    },
    "sensitivity_analyses": {
        "rna_clipping": {"quantiles": list(CLIP_QUANTILES)},
        "rna_scale_alignment": "CCLE-reference robust sample mapping with prespecified adaptive asinh",
        "protein_scale_harmonization": "TMT2 sample centering and label-free asinh/robust mapping",
        "tmt_scale_labels": "prespecified sensitivity labels only; never used to define the primary platform analysis",
    },
    "metrics": {
        "primary": "protein-wise Pearson correlation within each independent cohort",
        "secondary": ["Spearman correlation", "raw RMSE", "outcome-SD-normalized RMSE",
                      "cosine similarity", "calibration slope and intercept"],
        "normalized_rmse_formula": "RMSE(y, yhat) / SD(y); predictions are not separately standardized",
        "mse_thresholds": False,
    },
    "quality_control": {
        "max_sample_missing_fraction": MAX_SAMPLE_MISSING,
        "max_rna_feature_missing_fraction": MAX_RNA_FEATURE_MISSING,
        "max_protein_feature_missing_fraction": MAX_PROTEIN_FEATURE_MISSING,
        "external_protein_imputed": False,
    },
    "counts": {
        "ccle_total_samples": int(len(ccle_rna)),
        "ccle_train_samples": int(len(reference["train_samples"])),
        "ccle_test_samples": int(len(reference["test_samples"])),
        "rna_features": int(len(reference["rna_columns"])),
        "protein_models_per_family": int(len(reference["protein_columns"])),
        "external_cohorts": int(len(cohort_audit)),
        "external_samples_retained": int(cohort_audit["samples_retained"].sum()),
    },
    "cohort_specification": pv.COHORT_TABLE.to_dict("records"),
    "files": files,
}
manifest = pv.save_manifest(manifest_core, MANIFEST_PATH)
print("Manifest:", MANIFEST_PATH)
print("Manifest hash:", manifest["manifest_hash"])


## 7. Contract acceptance checklist


In [ ]:
checks = {
    "CCLE test split hash reconciled": reference["test_split_hash_sorted"] == split_info["test_split_hash_sorted"],
    "CCLE train split hash reconciled": pv.split_hash(reference["train_samples"], sorted_values=True) == split_info["train_split_hash_sorted"],
    "Exact ElasticNet inventory": int((model_inventory["display_model"] == "ElasticNet").sum()) == len(reference["protein_columns"]),
    "Exact XGBoost inventory": int((model_inventory["display_model"] == "XGBoost").sum()) == len(reference["protein_columns"]),
    "No external model selection": manifest["primary_analysis"]["external_model_selection"] is False,
    "One primary proxy": manifest["primary_analysis"]["proxy"].startswith("CCLE-training"),
    "Twenty cohorts": len(cohort_audit) == 20,
    "Primary independent of TMT1/TMT2": manifest["primary_analysis"]["tmt1_tmt2_used"] is False,
    "Primary no clipping": "no clipping" in manifest["primary_analysis"]["rna"],
    "Normalization QC figures generated": len(manifest["files"]["normalization_figures"]) == 5,
    "Normalization QC summary generated": Path(manifest["files"]["normalization_qc_summary"]["path"]).exists(),
    "External scTranslator removed": manifest["primary_analysis"]["external_sctranslator"] == "removed_incomplete",
    "MSE thresholds removed": manifest["metrics"]["mse_thresholds"] is False,
    "All contract files linked by manifest": (
        all(
            "sha256" in spec
            for key, spec in manifest["files"].items()
            if key not in {"cohort_contracts", "normalization_figures"}
        )
        and all("sha256" in spec for spec in manifest["files"]["cohort_contracts"].values())
        and all("sha256" in spec for spec in manifest["files"]["normalization_figures"].values())
    ),
}
acceptance = pd.DataFrame({"requirement": list(checks), "passed": list(checks.values())})
if not acceptance["passed"].all():
    raise RuntimeError("Contract acceptance failed:\n" + acceptance.to_string(index=False))
acceptance
